In [1]:
import os

os.environ["GEMINI_API_KEY"] = "AI-"

## Install libraries

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai \
               faiss-cpu tiktoken python-dotenv google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth=

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import DeterministicFakeEmbedding
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_5099/743834144.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DeterministicFakeEmbedding


## Step 1a - Indexing (Document Ingestion)

In [11]:
from youtube_transcript_api import YouTubeTranscriptApi


In [35]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "CZUtIIK4fdw"

try:
    # Instantiate the client
    yt_api = YouTubeTranscriptApi()

    # Fetch the transcript
    transcript_list = yt_api.fetch(video_id, languages=["en"])

    # Combine snippet text
    transcript = " ".join(snippet.text for snippet in transcript_list)
    print(transcript[:500] + "...")

except Exception as e:
    print(f"Error fetching transcript: {e}")

After my last video on how cursor 
works and why SpaceX is looking at it,   many of you asked for a proper 
deep dive. So in this video,   I'll show you how engineers actually use 
cursor end to end, including workflows,   mental models, and 25 practical tips you 
can apply immediately. Let's get into it. Here is what we are covering today. We start with 
what cursor actually is and how it thinks. So you   have the right mental model from the start. 
Then we cover the three interaction models,  ...


In [36]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='After my last video on how cursor\xa0\nworks and why SpaceX is looking at it,\xa0\xa0', start=0.16, duration=4.16), FetchedTranscriptSnippet(text='many of you asked for a proper\xa0\ndeep dive. So in this video,\xa0\xa0', start=4.32, duration=3.12), FetchedTranscriptSnippet(text="I'll show you how engineers actually use\xa0\ncursor end to end, including workflows,\xa0\xa0", start=7.44, duration=4.72), FetchedTranscriptSnippet(text="mental models, and 25 practical tips you\xa0\ncan apply immediately. Let's get into it.", start=12.16, duration=11.28), FetchedTranscriptSnippet(text='Here is what we are covering today. We start with\xa0\nwhat cursor actually is and how it thinks. So you\xa0\xa0', start=23.44, duration=5.12), FetchedTranscriptSnippet(text='have the right mental model from the start.\xa0\nThen we cover the three interaction models,\xa0\xa0', start=28.56, duration=4.48), FetchedTranscriptSnippet(text='tab, command K, 

## Step 1b - Indexing (Text Splitting)

In [37]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [38]:
len(chunks)

36

In [40]:
chunks[30]

Document(metadata={}, page_content='on your laptop. No cloud and no GitHub needed. So,\xa0\xa0 if your team has ever blocked a cloud AI tool for\xa0\nsecurity reasons, this is the one to look at. The\xa0\xa0 link is in the description and you can sign up\xa0\nthere. Moving on to Bugbot, which is one of my\xa0\xa0 favorite features in Cursor. Bugbot connects\xa0\nto your GitHub repository and automatically\xa0\xa0 reviews every pull request you open. It scans the\xa0\nchanges, identifies potential bugs, and leaves\xa0\xa0 comments directly on the PR exactly like a human\xa0\nreviewer would. When it finds an issue, it adds\xa0\xa0 a fix and cursor link. You click it, cursor opens\xa0\nwith a pre-filled prompt pointing directly to the\xa0\xa0 problem. You fix it and push. Think about what\xa0\nthis means at scale. Every PR gets an immediate\xa0\xa0 review pass. Your human reviewers spend their time\xa0\non architecture and logic rather than catching\xa0\xa0 missing null checks and unhandl

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [41]:
# We will use a lightweight fake embedding class or alternative, but let's keep it simple
# using DeterministicFakeEmbedding to avoid requiring OpenAI API keys for embeddings
embeddings = DeterministicFakeEmbedding(size=768)
vector_store = FAISS.from_documents(chunks, embeddings)

In [42]:
vector_store.index_to_docstore_id

{0: 'a51afa29-c0b5-4117-a3e9-df54b1e2043c',
 1: '392a4bc8-1232-4255-9029-793581a9585e',
 2: '57f5d36b-6a8b-472a-b556-2fc47fe5b368',
 3: '06f8a5bb-6d39-4953-b856-43ce1e2d422d',
 4: '8dbd5ab7-f5d8-409c-8d19-93c7fc1ee39a',
 5: '559ecf19-e762-4f76-84aa-c2e6b4e739a1',
 6: 'de51feef-fdaf-4136-b62f-8034d147965b',
 7: 'd9bf499a-65b5-45b8-80d5-a494596487f5',
 8: 'ce3550a2-8201-4f39-8def-d3622daf1d07',
 9: 'b81aa944-e8fc-4d6f-83fb-b9b02d5cff8a',
 10: '91100a36-3ed0-4622-9c04-0e9c9e4823bc',
 11: '3c7a79a1-4617-476f-beed-4af08696cad8',
 12: '2f57ab89-27a2-4cff-94d5-95c98948a749',
 13: '66903195-5ff4-42ae-8471-1c5bcc6cb458',
 14: '64aca8c0-3e04-46f9-a532-65bc74bde092',
 15: 'f041ec66-30ce-4f6a-bcd6-408478abf4b8',
 16: '613ab178-79f1-464b-8cd8-caa1a612bc99',
 17: '2661491c-bc95-49fd-8b67-75d1c4c7b7df',
 18: '5a057ed6-3a76-48f7-ab64-12b3fb7f498c',
 19: 'c0b1fece-abfa-4a04-b116-a3733558668b',
 20: '83039491-2776-498f-8a40-ca93c2ebb8cc',
 21: '073b54cd-d579-4b79-b522-0de682e78068',
 22: '6b4126d1-b9b0-

In [43]:
vector_store.get_by_ids(['a51afa29-c0b5-4117-a3e9-df54b1e2043c'])

[Document(id='a51afa29-c0b5-4117-a3e9-df54b1e2043c', metadata={}, page_content="After my last video on how cursor\xa0\nworks and why SpaceX is looking at it,\xa0\xa0 many of you asked for a proper\xa0\ndeep dive. So in this video,\xa0\xa0 I'll show you how engineers actually use\xa0\ncursor end to end, including workflows,\xa0\xa0 mental models, and 25 practical tips you\xa0\ncan apply immediately. Let's get into it. Here is what we are covering today. We start with\xa0\nwhat cursor actually is and how it thinks. So you\xa0\xa0 have the right mental model from the start.\xa0\nThen we cover the three interaction models,\xa0\xa0 tab, command K, and chat. Each one has a specific\xa0\nuse case, and knowing when to reach for which one\xa0\xa0 saves you a lot of time. Then we go deep on plan\xa0\nmode and agent mode. The two-phase workflow that\xa0\xa0 separates professional usage from just winging\xa0\nit with AI. Then rules, commands, and memories,\xa0\xa0 the most underused feature in cur

## Step 2 - Retrieval

In [44]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [45]:
retriever

VectorStoreRetriever(tags=['FAISS', 'DeterministicFakeEmbedding'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d1dc6a9af90>, search_kwargs={'k': 4})

In [46]:
retriever.invoke('What is Cursor\'s mcp')

[Document(id='3e560648-4edf-40e8-86f1-051f031f0eda', metadata={}, page_content='done. After that, add MCP servers incrementally,\xa0\xa0 not all at once. Start with the tools you use\xa0\nevery day. GitHub for PR management, notion\xa0\xa0 if that is what your specs live, playright if you\xa0\nwrite end to end test. Each one removes a context\xa0\xa0 switch from your workflow. The last thing I want\xa0\nto talk about is hooks. A hook lets you run your\xa0\xa0 own scripts at specific points in the agent loop.\xa0\nAfter every file edit, before a command runs, and\xa0\xa0 after a commit. Our project is now at about 80%.\xa0\nPagionation is in. Tests are written. The counter\xa0\xa0 bug is fixed. Bugbot is catching issues on every\xa0\nPR and the agent is pulling current documentation\xa0\xa0 through context 7. That is a realistic end state\xa0\nfor a real project using a real workflow. Your job\xa0\xa0 as a developer is no longer to write every line\xa0\nof code. Your job is to give the 

## Step 3 - Augmentation

In [47]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [48]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [49]:
question          = "give me summery of the topic"
retrieved_docs    = retriever.invoke(question)

In [50]:
retrieved_docs

[Document(id='64aca8c0-3e04-46f9-a532-65bc74bde092', metadata={}, page_content="to plan mode and describe what I want. Cursor does\xa0\xa0 not write a single line of code yet. Instead it\xa0\ngenerates an architecture overview. The database\xa0\xa0 schema changes and a complete checklist of every\xa0\nstep it plans to take. I can read through this,\xa0\xa0 edit it, add a to-do item, remove something that\xa0\nis out of scope and adjust the approach. The\xa0\xa0 plan is something I own and I can change before\xa0\nanything gets touched. Once the plan looks right,\xa0\xa0 I hit build. That is the moment cursor switches\xa0\nautomatically into agent mode and starts\xa0\xa0 executing. It works through the checklist, writes\xa0\nthe database migration, updates the task model,\xa0\xa0 adds the API endpoints, updates the front end,\xa0\nand checks off each item as it goes. I'm not\xa0\xa0 doing anything except watching and occasionally\xa0\nreviewing. Now, here is something important to\xa0\x

In [51]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"to plan mode and describe what I want. Cursor does\xa0\xa0 not write a single line of code yet. Instead it\xa0\ngenerates an architecture overview. The database\xa0\xa0 schema changes and a complete checklist of every\xa0\nstep it plans to take. I can read through this,\xa0\xa0 edit it, add a to-do item, remove something that\xa0\nis out of scope and adjust the approach. The\xa0\xa0 plan is something I own and I can change before\xa0\nanything gets touched. Once the plan looks right,\xa0\xa0 I hit build. That is the moment cursor switches\xa0\nautomatically into agent mode and starts\xa0\xa0 executing. It works through the checklist, writes\xa0\nthe database migration, updates the task model,\xa0\xa0 adds the API endpoints, updates the front end,\xa0\nand checks off each item as it goes. I'm not\xa0\xa0 doing anything except watching and occasionally\xa0\nreviewing. Now, here is something important to\xa0\xa0 understand about how this works. The code is being\xa0\nwritten into your ac

In [52]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [53]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      to plan mode and describe what I want. Cursor does\xa0\xa0 not write a single line of code yet. Instead it\xa0\ngenerates an architecture overview. The database\xa0\xa0 schema changes and a complete checklist of every\xa0\nstep it plans to take. I can read through this,\xa0\xa0 edit it, add a to-do item, remove something that\xa0\nis out of scope and adjust the approach. The\xa0\xa0 plan is something I own and I can change before\xa0\nanything gets touched. Once the plan looks right,\xa0\xa0 I hit build. That is the moment cursor switches\xa0\nautomatically into agent mode and starts\xa0\xa0 executing. It works through the checklist, writes\xa0\nthe database migration, updates the task model,\xa0\xa0 adds the API endpoints, updates the front end,\xa0\nand checks off each item as it goes. I'm not\xa

## Step 4 - Generation

In [54]:
answer = llm.invoke(final_prompt)
print(answer.content)

This transcript provides a deep dive into how engineers use "Cursor," covering its workflows, mental models, and practical tips. It explains the three interaction models (tab, command K, and chat) and details the two-phase workflow of "plan mode" (where an architecture overview, database schema changes, and a checklist are generated and can be edited) and "agent mode" (where Cursor executes the plan, writing code directly to local files without needing cloud or GitHub).

The transcript also highlights "Bugbot," a Cursor feature that connects to GitHub, automatically reviews pull requests, identifies bugs, and suggests fixes. It emphasizes the importance of committing code before running the agent and mentions a built-in quality gate where a second model can review changes made by the first. Other features like rules, commands, and memories are also mentioned as productivity enhancers.


## Building a Chain

In [58]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [59]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [60]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [61]:
parallel_chain.invoke('what is cursor')

{'context': "noise and noise reduces our output quality.\xa0\xa0 There are three types of rules worth knowing\xa0\nabout. Project rules live in your git repo and\xa0\xa0 apply to every developer on the team when they\xa0\npull. User rules are personal preferences that\xa0\xa0 stay on your machine and never get committed. For\xa0\nteam conventions, always use project rules. Now,\xa0\xa0 let's talk about commands because this is\xa0\nwhere individual productivity becomes team\xa0\xa0 productivity. A command is a reusable instruction\xa0\nyou save as a markdown file and trigger it with a\xa0\xa0 slash. The best way to understand this is through\xa0\na concrete example from our project. I need to add\xa0\xa0 structured logging to multiple services, the task\xa0\nservice, the notification service, and the email\xa0\xa0 service. I could write a detailed prompt every\xa0\nsingle time explaining exactly how I want the\xa0\xa0 logging structured. Or I can write that prompt\xa0\nonce, save it as

In [62]:
parser = StrOutputParser()

In [63]:
main_chain = parallel_chain | prompt | llm | parser

In [64]:
main_chain.invoke('Can you summarize the video')

'The transcript discusses several features and workflows for using Cursor, an AI-powered development tool.\n\nKey takeaways include:\n*   **Cursor\'s AI Interaction Modes**:\n    *   **Tab**: For staying in flow, letting AI keep up with you.\n    *   **Command K**: For making precise changes in one place.\n    *   **Chat**: For reasoning or making changes across multiple files, like repeating a pattern (e.g., error handling) across services.\n*   **Plan Mode**: For building entire features end-to-end locally without cloud or GitHub. You describe the desired feature, and Cursor generates an architecture overview, database schema changes, and a complete checklist. Users can review and edit this plan before Cursor automatically switches to "agent mode" to execute it, writing code directly to files.\n*   **Bugbot**: A feature that connects to your GitHub repository, automatically reviews every pull request, identifies potential bugs, and leaves comments with suggested fixes and a Cursor li